# Core Feature Selection EDA

`PREPROCESSING.ipynb`에서 생성한 70개 feature 버전을 기준으로, 실제 시간 검증을 통해 더 작은 core feature set을 고릅니다.

판단 기준:

- 풍력 발전량과 직접 연결되는 풍속, U/V 바람 성분을 우선합니다.
- 기온, 습도, 기압은 보조 feature로 별도 비교합니다.
- 구름, 강수 등은 full set에는 남겨두되 core 후보에서는 제외합니다.
- 검증은 미래 예측 상황에 맞춰 `2024년 holdout`으로 진행합니다.


In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import polars as pl
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

PREPROCESSED_DIR = Path("/root/wind/preprocessed")
CORE_DIR = Path("/root/wind/preprocessed_core")
CORE_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}

VALID_START = "2024-01-01 00:00:00"
RANDOM_STATE = 42


## Load 70-Feature Data

각 그룹별 parquet는 이미 최근접 격자 기반으로 만들어져 있습니다.

- train: `kst_dtm`, target, feature columns
- test: `forecast_id`, `kst_dtm`, feature columns


In [ ]:
group_train_data = {
    target: pl.read_parquet(PREPROCESSED_DIR / f"train_{target}.parquet")
    for target in TARGETS
}
group_test_data = {
    target: pl.read_parquet(PREPROCESSED_DIR / f"test_{target}.parquet")
    for target in TARGETS
}

for target in TARGETS:
    train_df = group_train_data[target]
    test_df = group_test_data[target]
    feature_cols = [c for c in train_df.columns if c not in ["kst_dtm", target]]
    test_feature_cols = [c for c in test_df.columns if c not in ["forecast_id", "kst_dtm"]]

    assert feature_cols == test_feature_cols
    assert sum(train_df.null_count().row(0)) == 0
    assert sum(test_df.null_count().row(0)) == 0

    print(target, "train", train_df.shape, "test", test_df.shape, "features", len(feature_cols))


## Feature Set 정의

비교할 후보는 네 가지입니다.

- `wind_speed`: 풍속 파생변수 + 시간 변수만 사용
- `wind_uv`: `wind_speed`에 U/V 바람 성분 추가
- `wind_context`: `wind_uv`에 기온, 습도, 기압, 경계층 추가
- `full_70`: 현재 전처리된 모든 feature 사용


In [ ]:
def strip_target_prefix(col: str, target: str) -> str:
    prefix = f"{target}_"
    return col[len(prefix):] if col.startswith(prefix) else col


def select_features(cols: list[str], target: str, mode: str) -> list[str]:
    selected = []

    for c in cols:
        b = strip_target_prefix(c, target)
        is_time = c in [
            "hour", "month", "dayofyear", "weekday",
            "hour_sin", "hour_cos", "dayofyear_sin", "dayofyear_cos",
        ]

        if mode == "full_70":
            selected.append(c)
            continue

        # Compact time representation. Tree model can use raw month, but cyclic terms keep season/hour continuity.
        if is_time:
            if c in ["month", "hour_sin", "hour_cos", "dayofyear_sin", "dayofyear_cos"]:
                selected.append(c)
            continue

        # LDAPS: local near-site wind speed summaries across nearest grids.
        if b.startswith("ldaps_wind_speed_"):
            selected.append(c)
            continue

        # GFS: hub-height-near and synoptic wind speed fields.
        if b in [
            "gfs_wind_speed_80m_mean",
            "gfs_wind_speed_100m_mean",
            "gfs_wind_speed_pbl_mean",
            "gfs_wind_speed_850hpa_mean",
            "gfs_wind_speed_700hpa_mean",
            "gfs_surface_0_gust_mean",
        ]:
            selected.append(c)
            continue

        if mode in ["wind_uv", "wind_context"]:
            if b in [
                "ldaps_heightAboveGround_10_10u_mean",
                "ldaps_heightAboveGround_10_10v_mean",
                "ldaps_heightAboveGround_50_50MUmax_mean",
                "ldaps_heightAboveGround_50_50MUmin_mean",
                "ldaps_heightAboveGround_50_50MVmax_mean",
                "ldaps_heightAboveGround_50_50MVmin_mean",
                "gfs_heightAboveGround_80_u_mean",
                "gfs_heightAboveGround_80_v_mean",
                "gfs_heightAboveGround_100_100u_mean",
                "gfs_heightAboveGround_100_100v_mean",
                "gfs_planetaryBoundaryLayer_0_u_mean",
                "gfs_planetaryBoundaryLayer_0_v_mean",
                "gfs_planetaryBoundaryLayer_0_VRATE_mean",
                "gfs_isobaricInhPa_850_u_mean",
                "gfs_isobaricInhPa_850_v_mean",
                "gfs_isobaricInhPa_700_u_mean",
                "gfs_isobaricInhPa_700_v_mean",
            ]:
                selected.append(c)
                continue

        if mode == "wind_context":
            if b in [
                "ldaps_heightAboveGround_2_t_mean",
                "ldaps_heightAboveGround_2_dpt_mean",
                "ldaps_heightAboveGround_2_r_mean",
                "ldaps_heightAboveGround_2_q_mean",
                "ldaps_surface_0_sp_mean",
                "ldaps_meanSea_0_prmsl_mean",
                "ldaps_etc_0_blh_mean",
                "gfs_heightAboveGround_2_2t_mean",
                "gfs_heightAboveGround_2_2d_mean",
                "gfs_heightAboveGround_2_2r_mean",
                "gfs_heightAboveGround_2_2sh_mean",
                "gfs_surface_0_sp_mean",
                "gfs_meanSea_0_prmsl_mean",
            ]:
                selected.append(c)
                continue

    return selected


FEATURE_SET_MODES = ["wind_speed", "wind_uv", "wind_context", "full_70"]

for target in TARGETS:
    all_features = [c for c in group_train_data[target].columns if c not in ["kst_dtm", target]]
    print()
    print(target)
    for mode in FEATURE_SET_MODES:
        print(mode, len(select_features(all_features, target, mode)))


## 2024 Holdout 검증

랜덤 split은 미래 예측 문제에 맞지 않으므로, 2024년을 검증 구간으로 둡니다.

- `kpx_group_1`, `kpx_group_2`: 2022~2023 학습, 2024 검증
- `kpx_group_3`: 2023 학습, 2024 검증


In [ ]:
def evaluate_feature_set(target: str, mode: str) -> dict:
    df = group_train_data[target].sort("kst_dtm")
    all_features = [c for c in df.columns if c not in ["kst_dtm", target]]
    feature_cols = select_features(all_features, target, mode)

    train_df = df.filter(pl.col("kst_dtm") < pl.lit(VALID_START).str.strptime(pl.Datetime))
    valid_df = df.filter(pl.col("kst_dtm") >= pl.lit(VALID_START).str.strptime(pl.Datetime))

    X_train = train_df.select(feature_cols).to_numpy()
    y_train = train_df[target].to_numpy()
    X_valid = valid_df.select(feature_cols).to_numpy()
    y_valid = valid_df[target].to_numpy()

    model = HistGradientBoostingRegressor(
        loss="absolute_error",
        max_iter=350,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )

    start = time.time()
    model.fit(X_train, y_train)
    pred = np.clip(model.predict(X_valid), 0, CAPACITY_KWH[target])

    mae = mean_absolute_error(y_valid, pred)
    nmae = mae / CAPACITY_KWH[target] * 100

    return {
        "target": target,
        "mode": mode,
        "n_features": len(feature_cols),
        "train_rows": train_df.height,
        "valid_rows": valid_df.height,
        "mae": mae,
        "nmae_pct": nmae,
        "fit_seconds": time.time() - start,
    }


score_rows = []
for target in TARGETS:
    for mode in FEATURE_SET_MODES:
        row = evaluate_feature_set(target, mode)
        score_rows.append(row)
        print(
            row["target"], row["mode"],
            "features", row["n_features"],
            "MAE", round(row["mae"], 2),
            "NMAE", round(row["nmae_pct"], 4),
        )

score_df = pd.DataFrame(score_rows)
score_df.to_csv(CORE_DIR / "feature_set_validation_scores.csv", index=False)
score_df


## 판단

`wind_speed`는 feature 수가 가장 적지만 U/V 방향 정보가 빠져 성능 손실이 큽니다.

`wind_uv`는 `full_70` 대비 feature 수를 70개에서 40개로 줄이면서 검증 성능은 거의 유지합니다. 따라서 core 버전은 `wind_uv`로 저장합니다.


In [ ]:
CORE_MODE = "wind_uv"

summary = (
    score_df
    .pivot(index="target", columns="mode", values="nmae_pct")
    .reset_index()
)
summary["core_minus_full_nmae_pctp"] = summary[CORE_MODE] - summary["full_70"]
summary.to_csv(CORE_DIR / "core_vs_full_summary.csv", index=False)
summary


## 시각화

아래 그래프들은 feature 축소 판단을 보기 위한 EDA입니다.

- feature set별 NMAE 비교
- core와 full의 그룹별 차이
- feature 수 대비 검증 성능
- core feature 카테고리 구성
- 2024년 일부 구간의 실제값 vs 예측값


In [ ]:
import matplotlib.pyplot as plt

PLOT_DIR = CORE_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

mode_order = ["wind_speed", "wind_uv", "wind_context", "full_70"]
mode_labels = {
    "wind_speed": "wind speed\n23 features",
    "wind_uv": "wind + u/v\n40 features",
    "wind_context": "wind + context\n53 features",
    "full_70": "full\n70 features",
}
colors = {
    "wind_speed": "#8c8c8c",
    "wind_uv": "#1f77b4",
    "wind_context": "#ff7f0e",
    "full_70": "#2ca02c",
}

# Visualization cell can run before the parquet-saving cell.
feature_columns_by_target = {
    target: select_features(
        [c for c in group_train_data[target].columns if c not in ["kst_dtm", target]],
        target,
        CORE_MODE,
    )
    for target in TARGETS
}

# 1. 그룹별 feature set NMAE 막대그래프
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, target in zip(axes, TARGETS):
    sub = score_df[score_df["target"] == target].set_index("mode").loc[mode_order]
    ax.bar(
        [mode_labels[m] for m in mode_order],
        sub["nmae_pct"],
        color=[colors[m] for m in mode_order],
    )
    ax.set_title(target)
    ax.set_ylabel("NMAE (%)")
    ax.tick_params(axis="x", labelrotation=0)
    best_idx = sub["nmae_pct"].idxmin()
    ax.axhline(sub.loc[best_idx, "nmae_pct"], color="black", linewidth=1, linestyle="--", alpha=0.5)
fig.suptitle("Feature Set Validation NMAE by KPX Group")
fig.tight_layout()
fig.savefig(PLOT_DIR / "feature_set_nmae_by_group.png", dpi=150)
plt.show()

# 2. core - full 차이
fig, ax = plt.subplots(figsize=(8, 4))
summary_plot = summary.set_index("target").loc[TARGETS]
vals = summary_plot["core_minus_full_nmae_pctp"]
bar_colors = ["#d62728" if v > 0 else "#2ca02c" for v in vals]
ax.bar(vals.index, vals.values, color=bar_colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Core(wind_uv) - Full(70) NMAE Difference")
ax.set_ylabel("NMAE difference (%p)")
for i, v in enumerate(vals.values):
    ax.text(i, v + (0.006 if v >= 0 else -0.012), f"{v:+.4f}", ha="center", va="bottom" if v >= 0 else "top")
fig.tight_layout()
fig.savefig(PLOT_DIR / "core_minus_full_nmae.png", dpi=150)
plt.show()

# 3. feature 수 대비 성능
fig, ax = plt.subplots(figsize=(9, 5))
for target in TARGETS:
    sub = score_df[score_df["target"] == target].set_index("mode").loc[mode_order]
    ax.plot(sub["n_features"], sub["nmae_pct"], marker="o", linewidth=2, label=target)
    for mode, row in sub.iterrows():
        ax.annotate(mode, (row["n_features"], row["nmae_pct"]), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_title("Validation NMAE vs Number of Features")
ax.set_xlabel("Number of features")
ax.set_ylabel("NMAE (%)")
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "nmae_vs_feature_count.png", dpi=150)
plt.show()

# 4. core feature 카테고리 구성

def feature_category(col: str) -> str:
    if col in ["month", "hour_sin", "hour_cos", "dayofyear_sin", "dayofyear_cos"]:
        return "time"
    if "_ldaps_" in col and "wind_speed" in col:
        return "ldaps wind speed"
    if "_ldaps_" in col:
        return "ldaps u/v"
    if "_gfs_" in col and "wind_speed" in col:
        return "gfs wind speed"
    if "surface_0_gust" in col:
        return "gfs gust"
    if "_gfs_" in col:
        return "gfs u/v"
    return "other"

cat_rows = []
for target, cols in feature_columns_by_target.items():
    counts = pd.Series([feature_category(c) for c in cols]).value_counts().sort_index()
    for category, count in counts.items():
        cat_rows.append({"target": target, "category": category, "count": count})
cat_df = pd.DataFrame(cat_rows)
cat_pivot = cat_df.pivot(index="target", columns="category", values="count").fillna(0).loc[TARGETS]

fig, ax = plt.subplots(figsize=(10, 4))
cat_pivot.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_title("Core Feature Composition")
ax.set_ylabel("Feature count")
ax.set_xlabel("")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
fig.tight_layout()
fig.savefig(PLOT_DIR / "core_feature_composition.png", dpi=150)
plt.show()

# 5. 2024년 holdout 일부 구간 실제값 vs 예측값
#    위 score와 같은 모델 설정으로 core mode만 다시 학습합니다.
forecast_rows = []
for target in TARGETS:
    df = group_train_data[target].sort("kst_dtm")
    all_features = [c for c in df.columns if c not in ["kst_dtm", target]]
    feature_cols = select_features(all_features, target, CORE_MODE)

    train_df = df.filter(pl.col("kst_dtm") < pl.lit(VALID_START).str.strptime(pl.Datetime))
    valid_df = df.filter(pl.col("kst_dtm") >= pl.lit(VALID_START).str.strptime(pl.Datetime))

    model = HistGradientBoostingRegressor(
        loss="absolute_error",
        max_iter=350,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )
    model.fit(train_df.select(feature_cols).to_numpy(), train_df[target].to_numpy())
    pred = np.clip(model.predict(valid_df.select(feature_cols).to_numpy()), 0, CAPACITY_KWH[target])

    tmp = pd.DataFrame(valid_df.select(["kst_dtm", target]).to_dicts())
    tmp["prediction"] = pred
    tmp["target"] = target
    forecast_rows.append(tmp)

valid_pred_df = pd.concat(forecast_rows, ignore_index=True)
valid_pred_df.to_csv(CORE_DIR / "core_valid_predictions_2024.csv", index=False)

plot_start = pd.Timestamp("2024-01-01")
plot_end = pd.Timestamp("2024-01-15")
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for ax, target in zip(axes, TARGETS):
    sub = valid_pred_df[
        (valid_pred_df["target"] == target)
        & (valid_pred_df["kst_dtm"] >= plot_start)
        & (valid_pred_df["kst_dtm"] < plot_end)
    ]
    ax.plot(sub["kst_dtm"], sub[target], label="actual", linewidth=1.5)
    ax.plot(sub["kst_dtm"], sub["prediction"], label="prediction", linewidth=1.2, alpha=0.85)
    ax.set_title(target)
    ax.set_ylabel("kWh")
    ax.legend(loc="upper right")
fig.suptitle("Core Model Holdout Example: Actual vs Prediction")
fig.tight_layout()
fig.savefig(PLOT_DIR / "core_valid_actual_vs_pred_2024_jan01_15.png", dpi=150)
plt.show()

print("plots saved to", PLOT_DIR)
print(sorted(p.name for p in PLOT_DIR.glob("*.png")))


## Core Parquet 저장

`/root/wind/preprocessed_core`에 그룹별 core train/test parquet를 저장합니다.


In [ ]:
feature_columns_by_target = {}

for target in TARGETS:
    train_df = group_train_data[target]
    test_df = group_test_data[target]
    all_features = [c for c in train_df.columns if c not in ["kst_dtm", target]]
    core_features = select_features(all_features, target, CORE_MODE)

    train_core = train_df.select(["kst_dtm", target, *core_features])
    test_core = test_df.select(["forecast_id", "kst_dtm", *core_features])

    assert train_core.width == len(core_features) + 2
    assert test_core.width == len(core_features) + 2
    assert sum(train_core.null_count().row(0)) == 0
    assert sum(test_core.null_count().row(0)) == 0

    train_core.write_parquet(CORE_DIR / f"train_{target}.parquet")
    test_core.write_parquet(CORE_DIR / f"test_{target}.parquet")
    feature_columns_by_target[target] = core_features

    print(target, "train", train_core.shape, "test", test_core.shape, "features", len(core_features))

with open(CORE_DIR / "feature_columns_by_target.json", "w", encoding="utf-8") as f:
    json.dump(feature_columns_by_target, f, ensure_ascii=False, indent=2)

print("saved to", CORE_DIR)
print(sorted(p.name for p in CORE_DIR.glob("*")))


## 실행 결과 요약

2024년 holdout 기준 비교 결과입니다. 값은 NMAE(%)입니다.

| target | wind_speed 23개 | wind_uv 40개 | wind_context 53개 | full_70 70개 | core-full 차이 |
|---|---:|---:|---:|---:|---:|
| kpx_group_1 | 10.5389 | 9.6670 | 9.5502 | 9.5419 | +0.1251%p |
| kpx_group_2 | 10.4909 | 9.4851 | 9.4922 | 9.5151 | -0.0300%p |
| kpx_group_3 | 10.4774 | 9.7220 | 9.7025 | 9.7016 | +0.0204%p |

판단:

- `wind_speed`는 너무 많이 줄여서 U/V 방향 정보 손실이 큽니다.
- `wind_context`와 `full_70`은 일부 그룹에서 아주 조금 좋지만, feature 수가 늘어나는 만큼의 이득은 작습니다.
- `wind_uv`는 70개를 40개로 줄이면서 성능 손실이 작고, `kpx_group_2`에서는 오히려 더 좋습니다.

따라서 core 전처리 파일은 `wind_uv` 40개 feature로 저장했습니다.


## Core Feature 목록 확인


In [ ]:
for target, cols in feature_columns_by_target.items():
    print()
    print(target, len(cols))
    for c in cols:
        print("-", c)
